## Mode of Inheritance (MOI) and evidence level information from Australian PanelApp sources

1. Mendeliome panel (https://panelapp-aus.org/panels/137/) 
2. Incidentalome panel (https://panelapp-aus.org/panels/126/) 
3. Additional findings – Paediatric panel (https://panelapp-aus.org/panels/3302/)

In [1]:
import pandas as pd


# CONFIG
tsv_file = "CAPNGSETB24-modified proband-leukoencephalopathy_freq.tsv"

mendeliome_file   = "./annotation/Mendeliome.tsv"
incidental_file   = "./annotation/Incidentalome.tsv"
paediatric_file   = "./annotation/Additional findings_Paediatric.tsv"

output_file = "result/freq_with_PanelApp_columns.tsv"


# Funtion create_panel_dict: Lookup dictionary from PanelApp
def create_panel_dict(panel_file, gene_col='Gene Symbol', status_col='GEL_Status', moi_col='Model_Of_Inheritance'):
    df_panel = pd.read_csv(panel_file, sep='\t', low_memory=False)
    
    def map_status(s):
        if pd.isna(s):
            return "Unknown"
        try:
            s = int(s)
            if s == 3:
                return "Green"
            elif s == 2:
                return "Amber"
            else:
                return "Red/Low"
        except:
            return str(s)
    
    # Create label
    df_panel['label'] = (
        df_panel[moi_col].astype(str) + " (" + df_panel[status_col].apply(map_status) + ")"
    )
    
    df_panel = df_panel.dropna(subset=[gene_col])
    return dict(zip(df_panel[gene_col], df_panel['label']))


# Load all three panels and create dictionaries
print("Loading PanelApp files...")

mendeliome_dict   = create_panel_dict(mendeliome_file)
incidental_dict   = create_panel_dict(incidental_file)
paediatric_dict   = create_panel_dict(paediatric_file)

# Load the original variant file
df = pd.read_csv(tsv_file, sep='\t')
print(f"\nOriginal variants loaded: {len(df)} rows")


# Function get_panel_status: Lookup gene(s) in the panel dictionary
def get_panel_status(gene_str, panel_dict, panel_name):
    if pd.isna(gene_str) or str(gene_str).strip() == '':
        return "."   # no found
    
    genes = [g.strip() for g in str(gene_str).split(',')]
    results = []
    for g in genes:
        if g in panel_dict:
            results.append(panel_dict[g])
        else:
            results.append(".")
    
    # All genes not found → return "."
    if not results:
        return "."
    
    return "; ".join(results)


# Add columns for each panel
print("\nAdding columns...")

df['Mode of Inheritance (Mendeliome)'] = df['Gene'].apply(
    lambda g: get_panel_status(g, mendeliome_dict, "Mendeliome")
)

df['Incidentalome Status'] = df['Gene'].apply(
    lambda g: get_panel_status(g, incidental_dict, "Incidentalome")
)

df['Paediatric Additional Status'] = df['Gene'].apply(
    lambda g: get_panel_status(g, paediatric_dict, "Paediatric Additional")
)

# Quick evidence columns (Green/Amber/Red/Unknown/.)
df['Mendeliome Evidence'] = df['Mode of Inheritance (Mendeliome)'].str.extract(r'\(([^)]+)\)')
df['Incidentalome Evidence'] = df['Incidentalome Status'].str.extract(r'\(([^)]+)\)')
df['Paediatric Evidence'] = df['Paediatric Additional Status'].str.extract(r'\(([^)]+)\)')

# Fill NaN with "." for evidence columns
for col in ['Mendeliome Evidence', 'Incidentalome Evidence', 'Paediatric Evidence']:
    df[col] = df[col].fillna(".")


# Save and show results 
df.to_csv(output_file, sep='\t', index=False)
print(f"\nFinished. Output saved to: {output_file}")

print("\nSample of added columns (first 12 rows):")
print(df[['Gene', 
          'Mode of Inheritance (Mendeliome)', 
          'Incidentalome Status', 
          'Paediatric Additional Status',
          'Mendeliome Evidence',
          'Incidentalome Evidence',
          'Paediatric Evidence']].head(12).to_string(index=False))

Loading PanelApp files...


FileNotFoundError: [Errno 2] No such file or directory: './annotation/Mendeliome.tsv'

## Generate HGMD batch input (hg19 genomic format)

format: chr1:45975088

In [ ]:
hgmd_output_file = "result/HGMD_batch_simple.txt"

# Create chromosome with "chr" prefix
df['Chr_with_prefix'] = 'chr' + df['Chr'].astype(str).str.replace(r'^chr', '', regex=True, flags=0)

# Create the position string
df['HGMD_input'] = df['Chr_with_prefix'] + ':' + df['Coordinate'].astype(str)

# Save to file
df['HGMD_input'].to_csv(
    hgmd_output_file,
    index=False,
    header=False,
    quoting=3,
    lineterminator='\n'
)

print(f"HGMD batch file created: {hgmd_output_file}")
print(f"Total lines written: {len(df)}")
# Show preview of first few lines
print("\nFirst 12 lines of HGMD input file (for verification):")
print(df['HGMD_input'].head(12).to_string(index=False))

HGMD batch file created: result/HGMD_batch_simple.txt
Total lines written: 29

First 12 lines of HGMD input file (for verification):
  chr1:45975088
 chr1:161184059
 chr2:136664812
 chr2:136664812
  chr5:88018388
 chr15:65295300
  chr1:44871049
 chr1:120612168
 chr1:120612194
chr11:118955882
 chr15:89876827
 chr2:207014657


## REVEL score

### Run this in terminal
- unzip revel-v1.3_all_chromosomes.zip

In [ ]:
import pandas as pd

# ====================== ADD REVEL SCORES ======================
revel_file = "./annotation/revel_with_transcript_ids"
genome_build = "hg19"

print("\nLoading REVEL scores...")
revel_df = pd.read_csv(
    revel_file,
    usecols=['chr', 'hg19_pos', 'grch38_pos', 'ref', 'alt', 'REVEL'],
    dtype={'chr': str, 'ref': str, 'alt': str},
    low_memory=False
)


Loading REVEL scores...


In [ ]:
print(revel_df.head())

  chr  hg19_pos grch38_pos ref alt  REVEL
0   1     35142      35142   G   A  0.027
1   1     35142      35142   G   C  0.035
2   1     35142      35142   G   T  0.043
3   1     35143      35143   T   A  0.018
4   1     35143      35143   T   C  0.034


In [ ]:
print(df.head())

     Gene Transcript HGNC      Transcript                      Variant  Chr  \
0  MMACHC          MMACHC     NM_015506.2                   G>G/GTTTTT    1   
1  NDUFS2          NDUFS2     NM_004550.4                      C>C/CTG    1   
2    DARS            DARS     NM_001349.2  C>CTTTTTTTTTTTTT/CTTTTTTTTT    2   
3    DARS            DARS     NM_001349.2  C>CTTTTTTTTT/CTTTTTTTTTTTTT    2   
4   MEF2C           MEF2C  NM_001193347.1                      GA>GA/G    5   

   Coordinate       Type Genotype Filters  Quality  ... Incidentalome Status  \
0    45975088  insertion      het    PASS    47.68  ...                    .   
1   161184059  insertion      het    PASS    49.29  ...                    .   
2   136664812  insertion      het    PASS   120.31  ...                    .   
3   136664812  insertion      het    PASS   120.31  ...                    .   
4    88018388   deletion      het    PASS    50.00  ...                    .   

                      Paediatric Additional 

In [ ]:
import pandas as pd

# Assuming your dataframes are already loaded as revel_df and df

# Make sure types are consistent
revel_df['Chr'] = revel_df['chr'].astype(str)
revel_df['hg19_pos'] = revel_df['hg19_pos'].astype(int)
revel_df['ref'] = revel_df['ref'].astype(str).str.upper()
revel_df['alt'] = revel_df['alt'].astype(str).str.upper()

# Create matching key in revel_df
revel_df['variant_key'] = (
    revel_df['Chr'] + ':' +
    revel_df['hg19_pos'].astype(str) + ':' +
    revel_df['ref'] + '>' +
    revel_df['alt']
)

# Create the same key in your main df (only works for SNVs!)
# You need to parse the Genotype column — format is quite variable
def extract_snv_key(row):
    gt = row['Genotype']
    if '>' not in gt:
        return None
    if len(gt.split('>')) != 2:
        return None
    refalt, _ = gt.split('/', 1)   # take part before possible /G etc
    ref, alt = refalt.split('>')
    if len(ref) == 1 and len(alt) == 1:  # only SNVs
        return f"{row['Chr']}:{row['Coordinate Type']}:{ref}>{alt}"
    return None

df['variant_key'] = df.apply(extract_snv_key, axis=1)

# Now left join
df = df.merge(
    revel_df[['variant_key', 'REVEL']],
    on='variant_key',
    how='left'
)

# Optional: drop the helper column
# df = df.drop(columns=['variant_key'])

print(df[['Gene', 'Chr', 'Coordinate Type', 'Genotype', 'REVEL']].head(10))

In [ ]:
# Show first 12 rows with the most relevant columns
cols_to_show = ['Gene', 'Variant', 'Coordinate', 'REVEL_score']
print(df[cols_to_show].head(12).to_string(index=False))

output_file = "result/freq_with_REVEL.tsv"

df.to_csv(output_file, sep='\t', index=False)
print(f"Finished. REVEL_score column added at the end → {output_file}")

  Gene                         Variant  Coordinate REVEL_score
MMACHC                      G>G/GTTTTT    45975088         nan
NDUFS2                         C>C/CTG   161184059         nan
  DARS     C>CTTTTTTTTTTTTT/CTTTTTTTTT   136664812         nan
  DARS     C>CTTTTTTTTT/CTTTTTTTTTTTTT   136664812         nan
 MEF2C                         GA>GA/G    88018388         nan
 MTFMT                         C>CA/CA    65295300         nan
RNF220                     A>A/ACTGCTG    44871049         nan
NOTCH2                           C>C/T   120612168         nan
NOTCH2                           C>C/G   120612194         nan
  HMBS CTTTTTTTTTTTTT>CTTTTTTTTTTTTT/C   118955882         nan
  POLG                        T>T/TTGC    89876827         nan
NDUFS1                     TAAA>TAAA/T   207014657         nan
Finished. REVEL_score column added at the end → result/freq_with_REVEL.tsv


## AlphaMissense annotation (missense variants only)

In [ ]:
am_file = "AlphaMissense_hg19.tsv.gz"   # ← use your actual filename/path

print("Loading AlphaMissense (hg19 version)... this file is very large, be patient")

am = pd.read_csv(
    am_file,
    sep="\t",
    comment="#",                    # skips the copyright/license lines
    low_memory=False,
    # No need for usecols if you want all columns; or specify to save memory:
    # usecols=["#CHROM", "POS", "REF", "ALT", "am_pathogenicity", "am_class"]
)

# Rename the chromosome column (pandas reads it as '#CHROM')
am = am.rename(columns={"#CHROM": "CHROM"})

print("AlphaMissense columns:", am.columns.tolist())
print(am.head(3))

In [ ]:
# Make sure types are compatible
df['CHROM'] = df['CHROM'].astype(str)
df['POS']   = pd.to_numeric(df['POS'], errors='coerce').astype('Int64')
df['REF']   = df['REF'].astype(str)
df['ALT']   = df['ALT'].astype(str)

am['CHROM'] = am['CHROM'].astype(str)
am['POS']   = pd.to_numeric(am['POS'], errors='coerce').astype('Int64')
am['REF']   = am['REF'].astype(str)
am['ALT']   = am['ALT'].astype(str)

# Merge - keep all your original variants (left join)
df = df.merge(
    am[['CHROM', 'POS', 'REF', 'ALT', 'am_pathogenicity', 'am_class']],
    on=['CHROM', 'POS', 'REF', 'ALT'],
    how='left'
)

# Fill missing values (no match / non-missense / etc.)
df['am_pathogenicity'] = df['am_pathogenicity'].fillna('.')
df['am_class']         = df['am_class'].fillna('.')

# Optional: move the new columns to the very end
cols = [c for c in df.columns if c not in ['am_pathogenicity', 'am_class']]
cols += ['am_pathogenicity', 'am_class']
df = df[cols]